### This notebook has the function of providing a visualiztion for H2 and H3. 

In [ ]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print(f"Python path: {sys.path}")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

df = pd.read_csv("backtrader/experiment/results/comparison.csv")

BASELINE_COLOR = "#95a5a6"
SIG_COLOR      = "#2ecc71"
NOSIG_COLOR    = "#e74c3c"
COMBO_OBS      = "#3498db"
COMBO_EXP      = "#bdc3c7"

ModuleNotFoundError: No module named 'pandas'

In [ ]:
individual = ["baseline", "changes_1", "changes_2", "changes_3"]
df_ind = df[df["branch"].isin(individual)].set_index("branch").reindex(individual)

fig, ax = plt.subplots(figsize=(9, 5))

colors = []
for branch in individual:
    if branch == "baseline":
        colors.append(BASELINE_COLOR)
    elif df_ind.loc[branch, "h2_rejected_cpu"] == "True":
        colors.append(SIG_COLOR)
    else:
        colors.append(NOSIG_COLOR)

bars = ax.bar(
    individual,
    df_ind["mean_cpu_energy_J"],
    yerr=df_ind["std_cpu_energy_J"],
    color=colors,
    capsize=5,
    width=0.5,
    edgecolor="black",
    linewidth=0.5,
)

ax.set_ylabel("Mean CPU Energy (J)", fontsize=12)
ax.set_xlabel("Branch", fontsize=12)
ax.set_title("RQ2 — Mean CPU Energy per Branch (Individual Changes)", fontsize=13)
ax.set_xticklabels(["Baseline", "Branch 1\n(Literature)", "Branch 2\n(Caching)", "Branch 3\n(Static Analysis)"])

# annotate reduction %
for i, branch in enumerate(individual):
    if branch == "baseline":
        continue
    pct = df_ind.loc[branch, "pct_reduction_cpu"]
    p   = df_ind.loc[branch, "p_cpu"]
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    ax.text(i, df_ind.loc[branch, "mean_cpu_energy_J"] + df_ind.loc[branch, "std_cpu_energy_J"] + 2,
            f"−{pct:.2f}%\n{sig}", ha="center", fontsize=9)

legend_patches = [
    mpatches.Patch(color=BASELINE_COLOR, label="Baseline"),
    mpatches.Patch(color=SIG_COLOR,      label="Significant reduction (p < 0.05)"),
    mpatches.Patch(color=NOSIG_COLOR,    label="No significant reduction"),
]
ax.legend(handles=legend_patches, fontsize=9)
ax.set_ylim(0, df_ind["mean_cpu_energy_J"].max() * 1.18)
plt.tight_layout()
plt.savefig("backtrader/experiment/results/h2_individual_branches.pdf", dpi=300)
plt.savefig("backtrader/experiment/results/h2_individual_branches.png", dpi=300)
plt.show()
print("H2 individual branch chart saved.")

In [ ]:
import glob, os

raw_data = {}
for branch in individual:
    path = f"backtrader/experiment/results/{branch}/energy_runnext.csv"
    if os.path.exists(path):
        raw = pd.read_csv(path)
        raw = raw[raw["duration_s"] >= 10]  # exclude incomplete runs
        raw["cpu_energy_J"] = raw["cpu_energy_kWh"] * 3_600_000
        # IQR filter
        q1, q3 = raw["cpu_energy_J"].quantile([0.25, 0.75])
        iqr = q3 - q1
        raw = raw[(raw["cpu_energy_J"] >= q1 - 1.5 * iqr) & (raw["cpu_energy_J"] <= q3 + 1.5 * iqr)]
        raw_data[branch] = raw["cpu_energy_J"].values

fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot(
    [raw_data[b] for b in individual],
    labels=["Baseline", "Branch 1\n(Literature)", "Branch 2\n(Caching)", "Branch 3\n(Static Analysis)"],
    patch_artist=True,
    medianprops=dict(color="black", linewidth=2),
)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel("CPU Energy (J)", fontsize=12)
ax.set_xlabel("Branch", fontsize=12)
ax.set_title("RQ2 — Distribution of CPU Energy per Branch (30 runs)", fontsize=13)
plt.tight_layout()
plt.savefig("backtrader/experiment/results/h2_boxplot.pdf", dpi=300)
plt.savefig("backtrader/experiment/results/h2_boxplot.png", dpi=300)
plt.show()
print("H2 box plot saved.")

In [ ]:
combo_branches = ["changes_1_2", "changes_1_3", "changes_2_3", "changes_1_2_3"]
df_combo = df[df["branch"].isin(combo_branches)].set_index("branch").reindex(combo_branches)

labels = ["B1+B2", "B1+B3", "B2+B3", "B1+B2+B3"]
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_exp = ax.bar(x - width/2, df_combo["h3_expected_pct"], width,
                  label="Expected (additive sum)", color=COMBO_EXP, edgecolor="black", linewidth=0.5)
bars_obs = ax.bar(x + width/2, df_combo["h3_observed_pct"], width,
                  label="Observed", color=COMBO_OBS, edgecolor="black", linewidth=0.5)

# annotate effect type
for i, branch in enumerate(combo_branches):
    effect = df_combo.loc[branch, "h3_effect"]
    diff   = df_combo.loc[branch, "h3_difference_pct"]
    ax.text(i, max(df_combo.loc[branch, "h3_expected_pct"],
                   df_combo.loc[branch, "h3_observed_pct"]) + 0.3,
            f"{effect}\n({diff:+.2f}%)", ha="center", fontsize=8)

ax.set_ylabel("Energy Reduction (%)", fontsize=12)
ax.set_xlabel("Combination Branch", fontsize=12)
ax.set_title("RQ3 — Expected vs Observed Energy Reduction for Combination Branches", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(fontsize=10)
ax.set_ylim(0, df_combo[["h3_expected_pct", "h3_observed_pct"]].max().max() * 1.2)
plt.tight_layout()
plt.savefig("backtrader/experiment/results/h3_combinations.pdf", dpi=300)
plt.savefig("backtrader/experiment/results/h3_combinations.png", dpi=300)
plt.show()
print("H3 combination chart saved.")